# Storeage

In [ ]:
#| default_exp game/storeblob

In [ ]:
#| export
from fastcore.basics import patch

import random


what do we need to import

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
from fastlite import *
import fasthtml.components as fc
import httpx

In [ ]:
#| export
from importlib import resources
from pathlib import Path
import tempfile
import shutil

In [ ]:
#| export
#data
from collections import namedtuple
from dataclasses import dataclass,  field, asdict
from typing import List
from datetime import date, datetime

from enum import Enum

In [ ]:
#| export
from HexMagic.game.kingdom import GameBoard,Kingdom,TradeRoute,Terrain,StyleCSS, Hex, TerraDemo, Geology, DrainageBasins, CountryFlag, TerraDemo
from HexMagic.game.country import CountryDetails

I am looking to create a database that can work with our Gameboard class and the website. We will have multiple users playing a gameboard and we need to save and load that into webpages. We also need the particular interface a user is seeing (like are they dropped down to a country. are they showing trade routes or climate) so I think we are going to need a few tables and classes to work with FastLite. The following are links about it
# Fastlite and sqlite-utils

> Fastlite builds on top of apswutils and sqlite-utils; generally you will want to use a combination of these

## Docs

- [Fastlite docs](https://answerdotai.github.io/fastlite/index.html.md): Fastlite docs home page
- [apswutils docs](https://answerdotai.github.io/apswutils/): apswutils docs
- [sqlite-utils docs](https://sqlite-utils.datasette.io/en/stable/_sources/python-api.rst.txt): Full sqlite-utils documentation

## Optional

- [APSW tour](https://rogerbinns.github.io/apsw/_sources/example.rst.txt)

What questions do you have?

1. I expect users to load and save multiple games, but only have one that is active
2. Right now we are using the web and I want to minimize cookies. certainly some of the interface stuff might be better served there so figuring what is in a cookie (which very well could be navigation and overlay toggling) and what is on the server (I think sendeding the map back and forth would be bad) is the goal of this. So whenever the user interacts that would change the state of the gameboard we would update, but probably not the interface
3. I don't think it has to persist across sessions
4. Lets definitely keep the encode and decode. at somepoint we should consider large maps where we only bring in a portion, but that will be much, much later if at all. We need to first see about handling the size of map we have (kind of at largest 80x60 hexes)

## Helpers

In [ ]:
#| export
# Define the result types
LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

## Tables

In [ ]:
#| export
@dataclass
class User:
    username: str
    email: str
    created: int  # Unix timestamp
    sessionID: str
    id: int = None

@dataclass
class Game:
    user_id: int
    name: str
    board_data: str
    is_active: bool
    last_modified_by: int
    created: int  # Unix timestamp
    modified: int  # Unix timestamp
    id: int = None

@dataclass
class Autosave:
    game_id: int
    user_id: int
    board_data: str
    turn_number: int
    created: int  # Unix timestamp
    id: int = None

@dataclass
class Template:
    name: str
    terrain_data: str
    description: str
    created: int  # Unix timestamp
    id: int = None


In [ ]:
#| export
class HexServer:

    def __init__(self, custom_path=None):
        path = HexServer.get_db_path(custom_path)
        self.path = path
        self.createDB()

    def createDB(self):
        db = database(self.path)
        self.db = db

        # Create tables using dataclasses
        db.create(User, pk='id', if_not_exists=True, transform=True)
        db.create(Game, pk='id', if_not_exists=True, transform=True)
        db.create(Autosave, pk='id', if_not_exists=True, transform=True)
        db.create(Template, pk='id', if_not_exists=True, transform=True)

        # Store table references using db.t notation
        self.users = db.t.user
        self.games = db.t.game
        self.autosaves = db.t.autosave
        self.templates = db.t.template
        
        # Populate templates (after tables exist)
        self.populate_templates()


    @staticmethod
    def get_db_path(custom_path=None):
        """Get database path. Tries package data, falls back to user directory."""
        if custom_path:
            return custom_path
        
        # Try package data directory first
        try:
            db_dir = resources.files('HexMagic').joinpath('data/db')
            db_path = Path(db_dir) / 'hexmagic.db'
            # Test if writable
            db_path.parent.mkdir(parents=True, exist_ok=True)
            db_path.touch(exist_ok=True)
            return str(db_path)
        except (PermissionError, OSError):
            # Fallback to user directory
            data_dir = Path.home() / '.hexmagic' / 'data'
            data_dir.mkdir(parents=True, exist_ok=True)
            return str(data_dir / 'hexmagic.db')

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.db.close()

    
    @staticmethod
    def _get(obj, key):
        """Get attribute from dict or object."""
        return obj[key] if isinstance(obj, dict) else getattr(obj, key)


In [ ]:
#| export
#this might be something we don't export
@patch
def reset_database(self:HexServer):
    """Delete and recreate database. USE WITH CAUTION!"""
    db_path = Path(self.path)  # Convert string to Path
    
    if db_path.exists():
        db_path.unlink()

    self.createDB()


### Templates

In [ ]:
#| export
@patch
def get_template(self: HexServer, template_id):
    """Get terrain for a template. Returns LoadResult(terrain, status, context)."""
    row = self.db.execute(
        "SELECT terrain_data FROM template WHERE id = ?",
        [template_id]
    ).fetchone()
    
    if not row:
        return LoadResult(None, 'not_found', f'Template {template_id} not found')
    
    try:
        terrain = Terrain.decode(row[0])
        return LoadResult(terrain, 'loaded', None)
    except Exception as e:
        return LoadResult(None, 'corrupted', f'Decode error: {str(e)[:50]}')

In [ ]:
#| export
@patch
def populate_templates(self: HexServer):
    """Populate templates table if empty."""
    # Check if already populated
    count = self.db.execute("SELECT COUNT(*) FROM template").fetchone()[0]
    if count > 0:
        return
    
    now = int(datetime.now().timestamp())
    td = TerraDemo()
    
    template_maps = {
        'bayArea_map': ('Bay Area', 'San Francisco Bay region'),
        'california_map': ('California', 'Full California coast'),
        'maui_map': ('Maui', 'Hawaiian island'),
        'japan_korea_map': ('Japan & Korea', 'East Asian region'),
        'normandy_map': ('Normandy', 'D-Day beaches'),
        'hong_kong_map': ('Hong Kong', 'South China coast'),
        'sydney_map': ('Sydney', 'Australian harbor'),
    }
    
    for method_name, (display_name, description) in template_maps.items():
        terrain = getattr(td, method_name)()
        self.templates.insert({
            'name': display_name,
            'terrain_data': terrain.encode(),
            'description': description,
            'created': now
        })


@patch
def add_template(self: HexServer, name, terrain, description=''):
    """Add a new terrain template."""
    now = int(datetime.now().timestamp())
    return self.templates.insert({
        'name': name,
        'terrain_data': terrain.encode(),
        'description': description,
        'created': now
    })


In [ ]:
#| export
@patch
def list_templates(self: HexServer):
    """Return list of (id, name) tuples for dropdown."""
    return [(HexServer._get(t, 'id'), HexServer._get(t, 'name')) for t in self.templates(order_by='name')]


### Games

In [ ]:
#| export
@patch
def save_game(self: HexServer, user_id, board, name=None, is_template=False):
    """Save a game board, archive previous version, and make it active."""
    now = int(datetime.now().timestamp())
    
    if name is None:
        name = f"Game {now}"
    
    board_data = board.encode()
    
    with self.db.conn:  # Transaction
        # Get current active game for this user
        active = self.db.execute(
            "SELECT id, board_data FROM game WHERE user_id = ? AND is_active = 1",
            [user_id]
        ).fetchone()
        
        if active:
            game_id, old_board_data = active
            
            # Get current max turn number for this game
            max_turn = self.db.execute(
                "SELECT COALESCE(MAX(turn_number), 0) FROM autosave WHERE game_id = ?",
                [game_id]
            ).fetchone()[0]
            
            # Archive current state
            self.autosaves.insert({
                'game_id': game_id,
                'user_id': user_id,
                'board_data': old_board_data,
                'turn_number': max_turn + 1,
                'created': now
            })
            
            # Keep only last 10 autosaves
            self.db.execute("""
                DELETE FROM autosave 
                WHERE game_id = ? 
                AND id NOT IN (
                    SELECT id FROM autosave 
                    WHERE game_id = ? 
                    ORDER BY turn_number DESC 
                    LIMIT 10
                )
            """, [game_id, game_id])
            
            # Update existing game
            self.db.execute("""
                UPDATE game 
                SET board_data = ?, last_modified_by = ?, modified = ?
                WHERE id = ?
            """, [board_data, user_id, now, game_id])
            
            return game_id  # Return just the ID, not the dict
        
        else:
            # No active game - deactivate any others and create new
            self.db.execute(
                "UPDATE game SET is_active = 0 WHERE user_id = ?",
                [user_id]
            )
            
            result = self.games.insert({
                'user_id': user_id,
                'name': name,
                'board_data': board_data,
                'is_active': True,
                'last_modified_by': user_id,
                'created': now,
                'modified': now
            })
            
            
            # Extract just the ID from the result
            return result['id'] if isinstance(result, dict) else result


In [ ]:
#| export
@patch
def start_from_template(self: HexServer, session_id, template_id, game_name=None,
                        top_n=3, year=1900, gender=None, rounds=50, scale=None, num_lakes=None):
    """Returns SaveResult(game_id, status, context)."""
    result = self.get_template(template_id)
    
    if result.status != 'loaded':
        return SaveResult(None, result.status, result.context)
    
    terrain = result.data
    
    # Optional downsampling
    if scale is not None:
        terrain = terrain.downsample_climate(scale=scale)
    
    # Optional lake carving
    if num_lakes is not None:
        terrain.carve_to_ocean(num_lakes=num_lakes)
    
    # Create GameBoard
    board = GameBoard(terrain, top_n=top_n, year=year, gender=gender)
    board.expand_kingdoms(max_rounds=rounds)
    
    # Create trade routes
    for country in board.kingdoms:
        neighbors = country.find_adjacent_kingdoms(board.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            dest_settlement = board.kingdoms[dest].settlements[0]
            path = board.find_path_dijkstra(origin, dest_settlement)
            if path is not None:
                country.routes.append(TradeRoute(path, origin=origin))
    
    # Get template name
   
    template = self.templates[template_id]
    template_name = HexServer._get(template, 'name') if template else "Unknown"
    
    #template = self.templates[template_id]
    #template_name = template['name'] if template else "Unknown"
    
    name = game_name or f"Game from {template_name}"
    game_id = self.save_game_by_session(session_id, board, name)

    
    
    return SaveResult(game_id, 'created', None)

In [ ]:
#| export
@patch
def load_active_game(self: HexServer, user_id):
    """Load user's active game. Returns LoadResult(board, status, context)."""
    try:
        game = self.games.rows_where('user_id = ? AND is_active = 1', [user_id]).__next__()
        board = GameBoard.decode(HexServer._get(game, 'board_data'))
        return LoadResult(board, 'loaded', None)
    except StopIteration:
        return LoadResult(None, 'no_active_game', 'No active game found for this user')
    except Exception as e:
        return LoadResult(None, 'corrupted', f'Decode error: {str(e)}')


### User

In [ ]:
#| export
@patch
def get_or_create_user(self: HexServer, session_id, username=None, email=None):
    """Get user by session_id, create if doesn't exist. Returns user_id."""
    # Try to find existing user
    row = self.db.execute(
        "SELECT id FROM user WHERE sessionID = ?",
        [session_id]
    ).fetchone()
    
    if row:
        return row[0]
    
    # Create new user
    now = int(datetime.now().timestamp())
    result = self.users.insert({
        'username': username or f"User_{session_id}",
        'email': email or '',
        'created': now,
        'sessionID': session_id
    })
    
    # Extract id from result dict
    return result['id'] if isinstance(result, dict) else result


In [ ]:
#| export
@patch
def load_game_by_session(self: HexServer, session_id):
    """Load active game for user with this session. Returns LoadResult."""
    user_id = self.get_or_create_user(session_id)
    return self.load_active_game(user_id)

@patch
def save_game_by_session(self: HexServer, session_id, board, name=None):
    """Save game for user with this session."""
    user_id = self.get_or_create_user(session_id)
    return self.save_game(user_id, board, name)


## Lets see how we are doing

In [ ]:
#| export
class DatabaseDebugger:
    """Test harness for HexServer with automatic cleanup."""
    
    def __init__(self, keep_on_error=True):
        """
        Args:
            keep_on_error: If True, preserve database when tests fail
        """
        self.temp_dir = tempfile.mkdtemp(prefix='hexmagic_test_')
        self.db_path = os.path.join(self.temp_dir, 'test.db')
        self.server = HexServer(custom_path=self.db_path)
        self.keep_on_error = keep_on_error
        self.failed = False
        
    def test_templates_populated(self):
        """Test that templates are auto-populated."""
        templates = self.server.list_templates()
        assert len(templates) == 7, f"Expected 7 templates, got {len(templates)}"
        print("✓ Templates populated correctly")
        
   
        
    def test_user_creation(self):
        """Test user creation and retrieval."""
        session1 = "test_session_123"
        user_id1 = self.server.get_or_create_user(session1, username="TestUser")
        user_id2 = self.server.get_or_create_user(session1)
        assert user_id1 == user_id2, "Same session should return same user"
        print("✓ User creation works")
        

        
    
        
    
        
    def preserve_db(self, test_name):
        """Copy database to inspection directory."""
        inspect_dir = Path.home() / '.hexmagic' / 'test_failures'
        inspect_dir.mkdir(parents=True, exist_ok=True)
        
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        dest = inspect_dir / f"{test_name}_{timestamp}.db"
        
        shutil.copy2(self.db_path, dest)
        print(f"⚠ Database preserved at: {dest}")
        
    def close(self):
        """Clean up test database."""
        self.server.db.close()
        
        if not self.failed or not self.keep_on_error:
            shutil.rmtree(self.temp_dir)
            print("✓ Test database cleaned up")
        else:
            print(f"⚠ Test database kept at: {self.temp_dir}")
            
    def __enter__(self):
        return self
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            self.failed = True
        self.close()




In [ ]:
#| export
@patch
def test_get_template(self: DatabaseDebugger):
    """Test retrieving a template."""
    reply = self.server.get_template(1)
    assert reply.status == 'loaded', f"Failed to load template: {reply.status} {reply.context}"
    assert reply.data is not None
    print("✓ Template retrieval works")

In [ ]:
#| export
@patch
def test_game_from_template(self: DatabaseDebugger):
    """Test creating a game from template."""
    session = "game_test_session"
    reply = self.server.start_from_template(
        session, 1, "Test Game", top_n=2, rounds=10
    )
    assert reply.status == 'created', f"Failed to create game: {reply.status} {reply.context}"
    assert reply.id is not None
    print(f"✓ Game created with id {reply.id}")

In [ ]:
#| export
@patch
def test_save_and_load(self: DatabaseDebugger):
    """Test saving and loading a game."""
    global myBoard
    
    session = "save_load_test"
    temp_file_path = None
    
    try:
        # Create a game
        reply = self.server.start_from_template(session, 1, rounds=5)
        
        # Get the raw encoded string
        user_id = self.server.get_or_create_user(session)
        game = self.server.games.rows_where('user_id = ? AND is_active = 1', [user_id]).__next__()
        
        # Save to temp file for debugging
        
        temp_file = tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.txt', prefix='gameboard_')
        temp_file.write(game['board_data'])
        temp_file.close()
        temp_file_path = temp_file.name
        
        # Try to decode
        myBoard = GameBoard.decode(game['board_data'])
        assert myBoard is not None
        assert len(myBoard.kingdoms) > 0
        
        # Success - delete temp file
        os.unlink(temp_file_path)
        print("✓ Save and load works")
        
    except Exception as e:
        # Failure - keep temp file for debugging
        if temp_file_path:
            print(f"✗ Decode failed: {e}")
            print(f"⚠ Encoded data saved to: {temp_file_path}")
        raise


In [ ]:
#| export
@patch
def test_autosaves(self: DatabaseDebugger):
    """Test that autosaves are created and limited to 10."""
    session = "autosave_test"
    replySFT = self.server.start_from_template(session, 1, rounds=5)
    
    # Make 15 saves
    replyLG = self.server.load_game_by_session(session)
    board = replyLG.data  # Extract the board from LoadResult
    for i in range(15):
        self.server.save_game_by_session(session, board, f"Save {i}")
    
    # Check autosave count
    count = self.server.db.execute(
        "SELECT COUNT(*) FROM autosave WHERE game_id = ?", [replySFT.id]  # Use replySFT.id
    ).fetchone()[0]
    assert count <= 10, f"Too many autosaves: {count}"
    print(f"✓ Autosaves limited correctly ({count} saves)")


In [ ]:
#| export
@patch
def test_list_templates(self: DatabaseDebugger):
    """Test listing templates returns correct format and data."""
    templates = self.server.list_templates()
    assert len(templates) == 7, f"Expected 7 templates, got {len(templates)}"
    
    # Check we can find a known template name
    names = [t[1] for t in templates]
    assert 'California' in names, f"Expected 'California' in {names}"
    
    # Verify each tuple has valid id and name
    for tid, name in templates:
        assert isinstance(tid, int) and tid > 0
        assert isinstance(name, str) and len(name) > 0
    
    print(f"✓ list_templates works: {names}")


In [ ]:
#| export
@patch
def run_all(self: DatabaseDebugger):
    """Run all tests."""
    tests = [
        self.test_templates_populated,
        self.test_get_template,
        self.test_user_creation,
        self.test_game_from_template,
        self.test_save_and_load,
        self.test_autosaves,
        self.test_list_templates
    ]
    
    for test in tests:
        try:
            test()
        except AssertionError as e:
            self.failed = True
            print(f"✗ {test.__name__}: {e}")
            if self.keep_on_error:
                self.preserve_db(test.__name__)
            raise
        except Exception as e:
            self.failed = True
            print(f"✗ {test.__name__} crashed: {e}")
            if self.keep_on_error:
                self.preserve_db(test.__name__)
            raise
            
    print("\n✓ All tests passed!")

We need a better list template test since this failed 
```


Prompt: 166 (05:25:08 PM)

I get ^^^^^^^^^^^^^^^^^^^^^^^^ File "/tmp/ipykernel_3772/1982000181.py", line 7, in newgame templates = mainServer.list_templates() ^^^^^^^^^^^^^^^^^^^^^^^^^^^ File "/app/data/HexMagic/HexMagic/game/store.py", line 216, in list_templates return [(t['id'], t['name']) for t in self.templates(order_by='name')] ~^^^^^^ TypeError: 'Template' object is not subscriptable

Assistant: 177

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The error is in store.py, not in your calling code. The list_templates method is treating Template objects like dictionaries:
Copied!

return [(t['id'], t['name']) for t in self.templates(order_by='name')]

But Template is an object, so it needs attribute access instead:
Copied!

return [(t.id, t.name) for t in self.templates(order_by='name')]

The test might be passing with different data, or possibly isn't being run. Want to fix that line in store.py?
```

In [ ]:
# Run all tests
with DatabaseDebugger(keep_on_error=False) as debugger:
    debugger.run_all()




what is going on

Do we have a similar issue with this 
 File "/usr/local/lib/python3.12/site-packages/anyio/_backends/_asyncio.py", line 2502, in run_sync_in_worker_thread
    return await future
           ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/site-packages/anyio/_backends/_asyncio.py", line 986, in run
    result = context.run(func, *args)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4247/4175927047.py", line 7, in create_game
    result = mainServer.start_from_template(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/app/data/HexMagic/HexMagic/game/store.py", line 339, in start_from_template
    template_name = template['name'] if template else "Unknown"
                    ~~~~~~~~^^^^^^^^
TypeError: 'Template' object is not subscriptable

are we going to have similar issues with games

can you give me fixes

Did I get them all

Fix test_autosaves

is test_save_and_load correct?

will this cache the bad string into myBoard so I can see if I can debug decoding it

lets save the string to a tmp directory so that I can import it into another notebook

from collections import namedtuple

# Define the result types
LoadResult = namedtuple('LoadResult', ['data', 'status', 'context'])
SaveResult = namedtuple('SaveResult', ['id', 'status', 'context'])

@patch
def load_active_game(self: HexServer, user_id):
    """Load user's active game. Returns LoadResult(board, status, context)."""
    try:
        game = self.games.rows_where('user_id = ? AND is_active = 1', [user_id]).__next__()
        board = GameBoard.decode(game['board_data'])
        return LoadResult(board, 'loaded', None)
    except StopIteration:
        return LoadResult(None, 'no_active_game', 'No active game found for this user')
    except Exception as e:
        return LoadResult(None, 'corrupted', f'Decode error: {str(e)}')

@patch
def load_game_by_session(self: HexServer, session_id):
    """Load active game for user with this session. Returns LoadResult."""
    user_id = self.get_or_create_user(session_id)
    return self.load_active_game(user_id)

@patch
def get_template(self: HexServer, template_id):
    """Get terrain for a template. Returns LoadResult(terrain, status, context)."""
    row = self.db.execute(
        "SELECT terrain_data FROM template WHERE id = ?",
        [template_id]
    ).fetchone()
    
    if not row:
        return LoadResult(None, 'not_found', f'Template {template_id} not found')
    
    try:
        terrain = Terrain.decode(row[0])
        return LoadResult(terrain, 'loaded', None)
    except Exception as e:
        return LoadResult(None, 'corrupted', f'Decode error: {str(e)}')

@patch
def start_from_template(self: HexServer, session_id, template_id, game_name=None,
                        top_n=3, year=1900, gender=None, rounds=50, scale=None, num_lakes=None):
    """Returns SaveResult(game_id, status, context)."""
    result = self.get_template(template_id)
    
    if result.status != 'loaded':
        return SaveResult(None, result.status, result.context)
    
    terrain = result.data
    
    # Optional downsampling
    if scale is not None:
        terrain = terrain.downsample_climate(scale=scale)
    
    # Optional lake carving
    if num_lakes is not None:
        terrain.carve_to_ocean(num_lakes=num_lakes)
    
    # Create GameBoard
    board = GameBoard(terrain, top_n=top_n, year=year, gender=gender)
    board.expand_kingdoms(max_rounds=rounds)
    
    # Create trade routes
    for country in board.kingdoms:
        neighbors = country.find_adjacent_kingdoms(board.kingdoms)
        for dest in neighbors:
            origin = country.settlements[0]
            dest_settlement = board.kingdoms[dest].settlements[0]
            path = board.find_path_dijkstra(origin, dest_settlement)
            if path is not None:
                country.routes.append(TradeRoute(path, origin=origin))
    
    # Get template name
    template = self.templates[template_id]
    template_name = template['name'] if template else "Unknown"
    
    name = game_name or f"Game from {template_name}"
    game_id = self.save_game_by_session(session_id, board, name)
    
    return SaveResult(game_id, 'created', None)
